# Week 4 Assessment: Vegetarian Recipe and Allergen Assistant

## 1. Business Understanding

### WHAT — What are we building?

An LLM application that reads a vegetarian recipe and:

- Extracts ingredients, allergens, diet suitability, preparation time
  and difficulty.
- Suggests a substitution to remove one selected allergen.
- Identifies other allergens that remain and any new allergens
  introduced by the substitution.
- Uses a separate LLM judge call to assess whether the substitution
  removes the selected allergen and keeps the dish workable.

Vegetarian scope: no meat or fish. Dairy and eggs are allowed.
Vegan recipes are also included.

### WHY — Why is this useful?

Recipe information is often written as unstructured text.
The assistant organises it into consistent fields and helps users
explore ingredient substitutions.

The application provides recipe-level guidance. It cannot confirm
product contents or rule out cross-contamination.

### HOW — What approach will we use?

We will use our existing Azure OpenAI deployment, without training
or fine-tuning a model.

We will compare three prompting approaches:

1. Zero-shot: instructions without worked examples.
2. Few-shot: instructions with a small set of worked examples.
3. Structured checklist: explicit extraction and verification tasks
   before returning the final structured answer.

All approaches will use the same output schema and common rules.
Reusable prompts will live in prompts.py and be imported into
this notebook and the application.

Pydantic will validate the response structure and permitted values.
Correctness will be evaluated separately against labelled recipes.

### EXPECTED RESULT — What should the experiments tell us?

The comparison should show which prompting approach performs best on:

- Ingredient extraction.
- Allergen identification, especially avoiding missed allergens.
- Diet classification.
- Preparation time and difficulty.
- Valid structured output.
- Substitution quality.
- Response time and token usage, where available.

The LLM judge will use a fixed assessment rubric. We will manually
review examples because the judge can also make mistakes.

### DECISION — How will we choose what to deploy?

We will select the approach using development results, prioritising
correct allergen identification and suitable substitutions.

We will then evaluate the selected approach on a separate held-out
set without further tuning.

The application will use:
- Streamlit for the interface.
- FastAPI for the backend.
- Azure OpenAI for recipe analysis and judging.
- Docker Compose to run the frontend and backend.
- GitHub for code, documentation and reproducible setup.

### Handling missing or unsuitable input

- Missing preparation time will remain unknown.
- Difficulty will be identified as stated or estimated.
- Uncertain ingredients will be flagged rather than assumed safe.
- Recipes with no detected target allergens will not require a substitution.
- Non-vegetarian recipes will be flagged as outside the intended scope.

## 2. Load and Understand the Dataset

### WHAT
Load the development recipes and inspect their structure.

Each record contains:
- Input: recipe text and the allergen requested for removal.
- Reference: the expected information used to evaluate the LLM.

### WHY
We need to understand the data before writing prompts.
Keeping inputs separate from reference answers prevents us from
accidentally sending the answers to the LLM.

### HOW
Use Python's json module to load development.json.
Display a summary and inspect one complete example.

The test set remains unopened until the final evaluation.

### EXPECTED RESULT
We expect 13 development recipes, each containing input and
reference sections.

### DECISION
Confirm that the data loads correctly and the labels make sense
before defining the application's output schema.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# Dataset folder, relative to the notebook's working directory.
data_dir = Path("data") / "vegetarian_recipe_dataset_v1"
development_path = data_dir / "development.json"

# Show where Python is looking, to make path problems easy to diagnose.
print("Working directory:", Path.cwd())

if not development_path.is_file():
    raise FileNotFoundError(
        f"Cannot find: {development_path.resolve()}\n"
        "Check where you extracted the dataset folder."
    )

# Load development data only. Do not load the test set yet.
with development_path.open("r", encoding="utf-8") as file:
    development_recipes = json.load(file)

print("Development recipes loaded:", len(development_recipes))

# Build a readable overview from the reference labels.
overview = pd.DataFrame([
    {
        "recipe_id": recipe["recipe_id"],
        "recipe": recipe["reference"]["title"],
        "diet": recipe["reference"]["diet"],
        "allergens": ", ".join(recipe["reference"]["allergens"]) or "None detected",
        "target_allergen": recipe["input"]["target_allergen"],
        "prep_minutes": recipe["reference"]["prep_time_minutes"],
        "difficulty": recipe["reference"]["difficulty"],
    }
    for recipe in development_recipes
])

display(overview)

# Inspect the input and reference separately.
sample_recipe = development_recipes[0]

print("\n--- INPUT: information we can send to the LLM ---")
print(sample_recipe["input"]["recipe_text"])
print("\nRequested allergen:", sample_recipe["input"]["target_allergen"])

print("\n--- REFERENCE: kept separately for evaluation ---")
print(json.dumps(sample_recipe["reference"], indent=2, ensure_ascii=False))

Working directory: c:\Saraswathi-DS\ds-october-2026-week4
Development recipes loaded: 13


,recipe_id,recipe,diet,allergens,target_allergen,prep_minutes,difficulty
0,DEVELOPMENT_01,Cashew tomato gravy,vegan,nuts,nuts,25.0,easy
1,DEVELOPMENT_02,Vegetable egg fried rice,vegetarian,eggs,eggs,10.0,easy
2,DEVELOPMENT_03,Celery potato soup,vegan,celery,celery,10.0,easy
3,DEVELOPMENT_04,Mustard cabbage stir-fry,vegan,mustard,mustard,8.0,easy
4,DEVELOPMENT_05,Tofu broccoli skewers,vegan,soya,soya,15.0,medium
5,DEVELOPMENT_06,Yoghurt cucumber dip,vegetarian,milk,milk,10.0,easy
6,DEVELOPMENT_07,Peanut lime slaw,vegan,peanuts,peanuts,12.0,easy
7,DEVELOPMENT_08,Lupin savoury pancakes,vegan,lupin,lupin,10.0,medium
8,DEVELOPMENT_09,Butter cream mashed potato,vegetarian,milk,milk,10.0,easy
9,DEVELOPMENT_10,Almond wheat cookies,vegan,"cereals_containing_gluten, nuts",nuts,12.0,medium



--- INPUT: information we can send to the LLM ---
Cashew tomato gravy
Ingredients: 40 g cashews; 200 g tomatoes; 100 ml water; 1 tsp vegetable oil; 1 tsp cumin; salt.
Preparation time: 25 minutes.
Difficulty: easy.
Method: Soak cashews for 20 minutes during preparation. Blend with tomatoes and water, then simmer with oil, cumin and salt.

Requested allergen: nuts

--- REFERENCE: kept separately for evaluation ---
{
  "title": "Cashew tomato gravy",
  "ingredients": [
    "cashews",
    "tomatoes",
    "water",
    "vegetable oil",
    "cumin",
    "salt"
  ],
  "allergens": [
    "nuts"
  ],
  "possible_allergens": [],
  "diet": "vegan",
  "prep_time_minutes": 25,
  "difficulty": "easy",
  "main_allergen": "nuts",
  "substitution_required": true,
  "substitution_reference": {
    "required_replacements": [
      {
        "ingredient": "cashews",
        "replacement": "hulled sunflower seeds",
        "workability_reason": "Soak and blend thoroughly to thicken the gravy; texture may 

## 3. Basic Data Quality Checks

### WHAT
Check recipe counts, duplicate IDs, duplicate text, required sections
and whether each requested allergen exists in its reference labels.

### WHY
Incorrect or duplicated records could make our prompt comparison
misleading.

### HOW
Run structural checks on the development set and list records with
missing information or uncertain ingredients.

### EXPECTED RESULT
13 development recipes with unique IDs and recipe texts.
Missing or uncertain information may be intentional.

### DECISION
Fix structural problems before proceeding. Review intentional unknowns
separately rather than deleting them or filling them with guesses.

These checks validate structure and consistency, not the factual
correctness of every reference label.

In [2]:
# Confirm the expected count and required record sections.
assert len(development_recipes) == 13, "Expected 13 development recipes."

for recipe in development_recipes:
    assert {"recipe_id", "split", "input", "reference"} <= recipe.keys()
    assert recipe["split"] == "development"

    recipe_input = recipe["input"]
    reference = recipe["reference"]

    assert recipe_input["recipe_text"].strip(), "Recipe text is empty."

    # A requested allergen must be listed in the reference answer.
    target = recipe_input["target_allergen"]
    assert target is None or target in reference["allergens"], (
        f"Target allergen mismatch: {recipe['recipe_id']}"
    )

# Check for duplicate identifiers and exact duplicate recipe text.
recipe_ids = [recipe["recipe_id"] for recipe in development_recipes]
recipe_texts = [
    recipe["input"]["recipe_text"]
    for recipe in development_recipes
]

assert len(recipe_ids) == len(set(recipe_ids)), "Duplicate recipe IDs found."
assert len(recipe_texts) == len(set(recipe_texts)), "Duplicate recipe text found."

print("Structural checks passed for all 13 development recipes.")

# Show cases that need careful handling, rather than automatic filling.
review_cases = pd.DataFrame([
    {
        "recipe_id": recipe["recipe_id"],
        "recipe": recipe["reference"]["title"],
        "prep_missing": recipe["reference"]["prep_time_minutes"] is None,
        "difficulty_missing": recipe["reference"]["difficulty"] is None,
        "possible_allergens": recipe["reference"]["possible_allergens"],
        "notes": recipe["reference"]["annotation_notes"],
    }
    for recipe in development_recipes
    if (
        recipe["reference"]["prep_time_minutes"] is None
        or recipe["reference"]["difficulty"] is None
        or recipe["reference"]["possible_allergens"]
        or recipe["reference"]["diet"] == "uncertain"
    )
])

display(review_cases)

Structural checks passed for all 13 development recipes.


,recipe_id,recipe,prep_missing,difficulty_missing,possible_allergens,notes
0,DEVELOPMENT_11,Roasted aubergine dip,True,True,[],Do not infer preparation time or difficulty fr...
1,DEVELOPMENT_12,Vegetarian stock risotto,False,False,"[celery, cereals_containing_gluten, milk]",Stock is explicitly vegetarian; vegan status i...


## 4. Define the LLM Output Schema

### WHAT
Define one consistent response structure for every prompting technique.

### WHY
We need predictable fields for evaluation and for displaying results
in Streamlit. Using the same schema also makes prompt comparisons fair.

### HOW
Create Pydantic models in schemas.py and import them into the notebook.

Separate:
- Confirmed allergens from possible allergens.
- Original recipe information from suggested changes.
- Remaining allergens from newly introduced allergens.

Missing preparation time and difficulty must be null.
If no substitution is appropriate, substitution must be null.

### EXPECTED RESULT
Correctly structured responses pass validation. Missing fields,
unexpected fields and invalid values produce validation errors.

### DECISION
Use this schema for all three prompts and the FastAPI application.
Evaluate factual correctness separately against reference labels.

In [3]:
from schemas import RecipeAnalysis

sample_response = {
    "title": "Cucumber salad",
    "ingredients": ["cucumber", "lemon juice", "salt"],
    "allergens": [],
    "possible_allergens": [],
    "diet": "vegan",
    "prep_time_minutes": None,
    "difficulty": None,
    "main_allergen": None,
    "substitution": None,
    "notes": ["Preparation time and difficulty were not stated."],
}

validated_response = RecipeAnalysis.model_validate(sample_response)

print("Schema validation passed.")
print(validated_response.model_dump_json(indent=2))

Schema validation passed.
{
  "title": "Cucumber salad",
  "ingredients": [
    "cucumber",
    "lemon juice",
    "salt"
  ],
  "allergens": [],
  "possible_allergens": [],
  "diet": "vegan",
  "prep_time_minutes": null,
  "difficulty": null,
  "main_allergen": null,
  "substitution": null,
  "notes": [
    "Preparation time and difficulty were not stated."
  ]
}


## 5. Verify That Invalid Responses Are Rejected

### WHAT
Check that Pydantic rejects an invalid diet label, negative preparation
time and a missing required field.

### WHY
Validating a correct example is only half the check.
We also need to confirm that incorrect structures are caught.

### HOW
Change one field at a time in our prepared sample response.
Catch and display the resulting validation errors.

### EXPECTED RESULT
All three invalid examples should be rejected.

### DECISION
If these checks behave as expected, proceed to Azure connection setup.
These checks do not call the LLM or incur API usage.

In [4]:
from copy import deepcopy

from pydantic import ValidationError

# Copy our valid example so the original remains unchanged.
invalid_diet = deepcopy(sample_response)
invalid_diet["diet"] = "healthy"  # Not one of our permitted diet labels.

invalid_time = deepcopy(sample_response)
invalid_time["prep_time_minutes"] = -10  # Time cannot be negative.

missing_ingredients = deepcopy(sample_response)
del missing_ingredients["ingredients"]  # This field is required.

invalid_examples = {
    "Invalid diet label": invalid_diet,
    "Negative preparation time": invalid_time,
    "Missing ingredients field": missing_ingredients,
}

for description, response in invalid_examples.items():
    print(f"\n--- {description} ---")

    try:
        RecipeAnalysis.model_validate(response)
    except ValidationError as error:
        print("Rejected as expected.")
        for issue in error.errors():
            field = ".".join(str(part) for part in issue["loc"])
            print(f"  {field}: {issue['msg']}")
    else:
        raise AssertionError(f"Unexpectedly accepted: {description}")


--- Invalid diet label ---
Rejected as expected.
  diet: Input should be 'vegan', 'vegetarian', 'non_vegetarian' or 'uncertain'

--- Negative preparation time ---
Rejected as expected.
  prep_time_minutes: Input should be greater than or equal to 0

--- Missing ingredients field ---
Rejected as expected.
  ingredients: Field required


## 6. Verify the Azure OpenAI Connection

### WHAT
Load our existing Azure settings and make one small API request.

### WHY
Creating a client does not prove that authentication and the model
deployment work. An actual response confirms the connection.

### HOW
Load credentials from .env, check that required settings exist,
and request a short response from our deployed model.

Never print the API key or include it in notebook code.

### EXPECTED RESULT
Azure returns a short response without an authentication or
deployment error.

### DECISION
If successful, proceed to defining the three prompting techniques.
This check makes one API request and may incur a small usage cost.

In [5]:
import os

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

# Reuse the environment-loading approach from your class notebook.
env_path = find_dotenv(usecwd=True)

if not env_path:
    raise FileNotFoundError(
        "Could not find .env. Check that it is in your project folder."
    )

load_dotenv(env_path)

required_settings = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_DEPLOYMENT",
]

missing_settings = [
    name for name in required_settings
    if not os.getenv(name, "").strip()
]

if missing_settings:
    raise ValueError(
        "Missing environment settings: " + ", ".join(missing_settings)
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].strip().rstrip("/")
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"].strip()

# Same client configuration used in your class setup.
client = OpenAI(
    base_url=f"{endpoint}/openai/v1/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    timeout=60,
    max_retries=0,  # Keep this connection check to one attempt.
)

print("Required settings loaded.")
print("Deployment:", deployment)

# Make an actual request; credentials are never printed.
connection_response = client.chat.completions.create(
    model=deployment,
    messages=[
        {
            "role": "user",
            "content": "Reply with only the word CONNECTED.",
        }
    ],
)

print("Model response:", connection_response.choices[0].message.content)

Required settings loaded.
Deployment: gpt-5.4-mini
Model response: CONNECTED


## 7. Define Shared Instructions and the Zero-Shot Prompt

### WHAT
Create prompts.py to hold reusable prompt definitions.

Begin with shared rules and a zero-shot baseline.

### WHY
Centralised prompts are easier to maintain and can be reused by
both the notebook and FastAPI.

Shared rules keep the vegetarian scope, uncertainty handling and
output requirements consistent across prompting techniques.

### HOW
Use a system message for instructions and the Pydantic-generated
JSON schema. Put recipe input in a separate user message.

Treat recipe text as data, not as instructions to follow.

Zero-shot provides instructions but no worked examples.

### EXPECTED RESULT
A reusable function builds the messages for a recipe request.
Building messages does not call Azure.

### DECISION
Inspect the messages before making the first recipe-analysis request.

In [6]:
from prompts import build_zero_shot_messages

# Use the first development recipe without its reference answers.
recipe_input = development_recipes[0]["input"]

zero_shot_messages = build_zero_shot_messages(
    recipe_text=recipe_input["recipe_text"],
    target_allergen=recipe_input["target_allergen"],
)

print("Message roles:", [message["role"] for message in zero_shot_messages])

print("\n--- User message sent to the LLM ---")
print(zero_shot_messages[1]["content"])

Message roles: ['system', 'user']

--- User message sent to the LLM ---
{"recipe_text": "Cashew tomato gravy\nIngredients: 40 g cashews; 200 g tomatoes; 100 ml water; 1 tsp vegetable oil; 1 tsp cumin; salt.\nPreparation time: 25 minutes.\nDifficulty: easy.\nMethod: Soak cashews for 20 minutes during preparation. Blend with tomatoes and water, then simmer with oil, cumin and salt.", "target_allergen": "nuts"}


## 8. Run the First Zero-Shot Recipe Analysis

### WHAT
Send one development recipe to Azure and validate its response.

### WHY
Before processing multiple recipes, check that the prompt, model
and Pydantic schema work together on one real request.

### HOW
Use the messages prepared in Step 7.
Validate the returned JSON directly with RecipeAnalysis.
Record response time and token usage.

We will not automatically repair or retry an invalid response:
first-attempt failures are useful evaluation evidence.

### EXPECTED RESULT
A structured analysis of cashew tomato gravy, identifying nuts
and suggesting a vegetarian replacement for cashews.

### DECISION
Inspect the actual response before running further experiments.
One successful example does not establish overall prompt quality.

In [7]:
from time import perf_counter

from pydantic import ValidationError
from schemas import RecipeAnalysis

# One API request using our zero-shot messages.
start_time = perf_counter()

recipe_response = client.chat.completions.create(
    model=deployment,
    messages=zero_shot_messages,
)

elapsed_seconds = perf_counter() - start_time

# Preserve the original response for inspection.
raw_response = recipe_response.choices[0].message.content
zero_shot_result = None

print(f"Response time: {elapsed_seconds:.2f} seconds")
print("Finish reason:", recipe_response.choices[0].finish_reason)

if recipe_response.usage is not None:
    print("Input tokens:", recipe_response.usage.prompt_tokens)
    print("Output tokens:", recipe_response.usage.completion_tokens)

# Validate the response without stripping or repairing its content.
if not raw_response:
    print("\nNo text response returned.")
    print("Refusal:", getattr(recipe_response.choices[0].message, "refusal", None))
else:
    try:
        zero_shot_result = RecipeAnalysis.model_validate_json(raw_response)
    except ValidationError as error:
        print("\nResponse failed validation:")
        print(error)
        print("\nOriginal response:")
        print(raw_response)
    else:
        print("\nResponse passed schema validation.")
        print(zero_shot_result.model_dump_json(indent=2))

Response time: 2.20 seconds
Finish reason: stop
Input tokens: 2002
Output tokens: 305

Response passed schema validation.
{
  "title": "Cashew tomato gravy",
  "ingredients": [
    "cashews",
    "tomatoes",
    "water",
    "vegetable oil",
    "cumin",
    "salt"
  ],
  "allergens": [
    "nuts"
  ],
  "possible_allergens": [],
  "diet": "vegetarian",
  "prep_time_minutes": 25,
  "difficulty": "easy",
  "main_allergen": "nuts",
  "substitution": {
    "replacements": [
      {
        "ingredient": "cashews",
        "replacement": "sunflower seeds",
        "adjustment": "Use the same 40 g amount; soak if needed, then blend with the tomatoes and water to keep a similar creamy texture."
      }
    ],
    "explanation": "The confirmed allergen is nuts, selected because the target allergen was supplied and cashews are explicitly listed. Replace the cashews only; the remaining ingredients are kept as written. This adaptation remains vegetarian, but sunflower seeds are a new allergen ri

In [8]:
print("--- Proposed substitution ---")

if zero_shot_result.substitution is not None:
    print(zero_shot_result.substitution.model_dump_json(indent=2))
else:
    print("No substitution proposed.")

print("\n--- Notes ---")
for note in zero_shot_result.notes:
    print("-", note)

--- Proposed substitution ---
{
  "replacements": [
    {
      "ingredient": "cashews",
      "replacement": "sunflower seeds",
      "adjustment": "Use the same 40 g amount; soak if needed, then blend with the tomatoes and water to keep a similar creamy texture."
    }
  ],
  "explanation": "The confirmed allergen is nuts, selected because the target allergen was supplied and cashews are explicitly listed. Replace the cashews only; the remaining ingredients are kept as written. This adaptation remains vegetarian, but sunflower seeds are a new allergen risk for people sensitive to seeds, and the final dish may still contain other unlabelled traces depending on the product used.",
  "remaining_allergens": [],
  "introduced_allergens": [
    "sesame"
  ]
}

--- Notes ---
- The recipe is vegetarian because it contains no meat or fish; dairy and eggs are not used.
- Cashews are a confirmed nut allergen, so they are the main allergen.
- Preparation time is explicitly stated as 25 minutes.


### First Zero-Shot Result: Findings

The response passed Pydantic validation and correctly extracted the
ingredients, original nut allergen, preparation time and difficulty.

Two factual errors were identified:
1. The original recipe was labelled vegetarian instead of the more
   specific supported label, vegan.
2. Sunflower seeds were incorrectly mapped to the sesame allergen.

The proposed cashew-to-sunflower-seed substitution matches an acceptable
reference option, although texture may differ.

Decision: preserve this baseline response and leave the prompt unchanged.
Compare the other prompting techniques before drawing conclusions.

Schema validity does not establish factual correctness.

## 9. Build the Few-Shot Prompt

### WHAT
Provide three example recipes with prepared expected answers
before presenting the recipe we want analysed.

### WHY
Examples demonstrate how to apply our instructions consistently.
We will measure whether they improve results; improvement is
not guaranteed.

### HOW
Load three records from examples.json and convert their reference
labels into our RecipeAnalysis response format.

Construct messages in this order:
- System instructions.
- Three example user/assistant pairs.
- The development recipe to analyse.

Use the same shared instructions and output schema as zero-shot.
Do not use development or test reference answers as prompt examples.

### EXPECTED RESULT
Eight messages: one system message, six example messages and
one final user message.

### DECISION
Check the message structure before calling Azure.
This step does not make an API request.

In [10]:
import importlib
import prompts

importlib.reload(prompts)

with (data_dir / "examples.json").open("r", encoding="utf-8") as file:
    example_recipes = json.load(file)

# Use the first three prepared examples.
# The fourth ambiguity example is reserved for later review.
few_shot_examples = example_recipes[:3]

recipe_input = development_recipes[0]["input"]

few_shot_messages = prompts.build_few_shot_messages(
    recipe_text=recipe_input["recipe_text"],
    target_allergen=recipe_input["target_allergen"],
    examples=few_shot_examples,
)

print("Worked examples:")
for example in few_shot_examples:
    print("-", example["reference"]["title"])

print("\nNumber of messages:", len(few_shot_messages))
print("Message roles:", [message["role"] for message in few_shot_messages])

print("\nFinal user message:")
print(few_shot_messages[-1]["content"])

Worked examples:
- Creamy sweetcorn soup
- Tahini lemon dressing
- Cucumber mint salad

Number of messages: 8
Message roles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user']

Final user message:
{"recipe_text": "Cashew tomato gravy\nIngredients: 40 g cashews; 200 g tomatoes; 100 ml water; 1 tsp vegetable oil; 1 tsp cumin; salt.\nPreparation time: 25 minutes.\nDifficulty: easy.\nMethod: Soak cashews for 20 minutes during preparation. Blend with tomatoes and water, then simmer with oil, cumin and salt.", "target_allergen": "nuts"}


## 10. Run the Few-Shot Recipe Analysis

### WHAT
Send the few-shot request to Azure and validate its response.

### WHY
Check whether worked examples improve the answer for the same recipe.

### HOW
Keep the deployment, shared instructions, schema and recipe unchanged.
The additional worked examples are the experimental difference.

Record response time, token usage and first-attempt validation.
Compare the diet label and introduced allergens with the reference.

### EXPECTED RESULT
A valid response identifying the original recipe as vegan and nuts
as its confirmed allergen.

Introduced allergens depend on the proposed replacement:
plain sunflower seeds should not be labelled sesame.

### DECISION
Inspect the actual result. One recipe is only an initial check;
the full development set is needed to compare prompting techniques.

In [11]:
start_time = perf_counter()

few_shot_response = client.chat.completions.create(
    model=deployment,
    messages=few_shot_messages,
)

few_shot_seconds = perf_counter() - start_time
few_shot_raw = few_shot_response.choices[0].message.content
few_shot_result = None

print(f"Response time: {few_shot_seconds:.2f} seconds")
print("Finish reason:", few_shot_response.choices[0].finish_reason)

if few_shot_response.usage is not None:
    print("Input tokens:", few_shot_response.usage.prompt_tokens)
    print("Output tokens:", few_shot_response.usage.completion_tokens)

if not few_shot_raw:
    print("\nNo text response returned.")
    print(
        "Refusal:",
        getattr(few_shot_response.choices[0].message, "refusal", None),
    )
else:
    try:
        few_shot_result = RecipeAnalysis.model_validate_json(few_shot_raw)
    except ValidationError as error:
        print("\nResponse failed validation:")
        print(error)
        print("\nOriginal response:")
        print(few_shot_raw)
    else:
        print("\nResponse passed schema validation.")

        # Display key fields separately to reduce output truncation.
        print("Diet:", few_shot_result.diet)
        print("Ingredients:", few_shot_result.ingredients)
        print("Confirmed allergens:", few_shot_result.allergens)
        print("Possible allergens:", few_shot_result.possible_allergens)
        print("Preparation time:", few_shot_result.prep_time_minutes)
        print("Difficulty:", few_shot_result.difficulty)
        print("Selected allergen:", few_shot_result.main_allergen)

Response time: 2.00 seconds
Finish reason: stop
Input tokens: 2580
Output tokens: 151

Response passed schema validation.
Diet: vegan
Ingredients: ['cashews', 'tomatoes', 'water', 'vegetable oil', 'cumin', 'salt']
Confirmed allergens: ['nuts']
Possible allergens: []
Preparation time: 25
Difficulty: easy
Selected allergen: nuts


In [12]:
if few_shot_result is not None:
    print("--- Proposed substitution ---")

    if few_shot_result.substitution is not None:
        print(few_shot_result.substitution.model_dump_json(indent=2))
    else:
        print("No substitution proposed.")

    print("\n--- Notes ---")
    for note in few_shot_result.notes:
        print("-", note)
else:
    print("No validated result. Review the previous cell's output.")

--- Proposed substitution ---
{
  "replacements": [
    {
      "ingredient": "cashews",
      "replacement": "cooked white beans",
      "adjustment": "Use the same weight by volume as the cashews for body; soak time is not needed, and add a little extra water if the gravy is too thick."
    }
  ],
  "explanation": "Replace each listed source of the selected allergen while keeping the recipe vegetarian.",
  "remaining_allergens": [],
  "introduced_allergens": []
}

--- Notes ---


## 11. Build the Structured Checklist Prompt

### WHAT
Add a third prompting technique that explicitly asks the model
to check extraction, diet and substitution details before returning JSON.

### WHY
A checklist may reduce overlooked ingredients, inconsistent labels
and unclear substitution guidance. We must measure whether it does.

### HOW
Reuse the zero-shot system instructions, schema and input.
Append a checklist to the system message without adding examples.

Request only the final JSON, not a detailed reasoning transcript.

### EXPECTED RESULT
Two messages: one system instruction and one user recipe.

### DECISION
Check the request structure, then evaluate this technique using
the same recipes and settings as the other approaches.

In [13]:
# Reload the file to pick up the new prompt and function.
importlib.reload(prompts)

recipe_input = development_recipes[0]["input"]

checklist_messages = prompts.build_checklist_messages(
    recipe_text=recipe_input["recipe_text"],
    target_allergen=recipe_input["target_allergen"],
)

print("Number of messages:", len(checklist_messages))
print("Message roles:", [message["role"] for message in checklist_messages])

print("\nFinal user message:")
print(checklist_messages[-1]["content"])

Number of messages: 2
Message roles: ['system', 'user']

Final user message:
{"recipe_text": "Cashew tomato gravy\nIngredients: 40 g cashews; 200 g tomatoes; 100 ml water; 1 tsp vegetable oil; 1 tsp cumin; salt.\nPreparation time: 25 minutes.\nDifficulty: easy.\nMethod: Soak cashews for 20 minutes during preparation. Blend with tomatoes and water, then simmer with oil, cumin and salt.", "target_allergen": "nuts"}


## 12. Run the Structured Checklist Analysis

### WHAT
Send the checklist request to Azure and validate the response.

### WHY
Check whether explicit verification instructions improve extraction,
allergen reporting and substitution guidance.

### HOW
Use the same deployment, recipe and output schema as before.
Record response time, token usage and first-attempt validation.

### EXPECTED RESULT
A valid response with the correct vegan label, nut allergen and
a practical substitution with clear adjustment instructions.

Introduced allergens must match the actual proposed replacement.

### DECISION
Review the response before comparing all three techniques across
the full development set. This single recipe is an initial check.

In [14]:
start_time = perf_counter()

checklist_response = client.chat.completions.create(
    model=deployment,
    messages=checklist_messages,
)

checklist_seconds = perf_counter() - start_time
checklist_raw = checklist_response.choices[0].message.content
checklist_result = None

print(f"Response time: {checklist_seconds:.2f} seconds")
print("Finish reason:", checklist_response.choices[0].finish_reason)

if checklist_response.usage is not None:
    print("Input tokens:", checklist_response.usage.prompt_tokens)
    print("Output tokens:", checklist_response.usage.completion_tokens)

if not checklist_raw:
    print("\nNo text response returned.")
    print(
        "Refusal:",
        getattr(checklist_response.choices[0].message, "refusal", None),
    )
else:
    try:
        checklist_result = RecipeAnalysis.model_validate_json(checklist_raw)
    except ValidationError as error:
        print("\nResponse failed validation:")
        print(error)
        print("\nOriginal response:")
        print(checklist_raw)
    else:
        print("\nResponse passed schema validation.")
        print("Diet:", checklist_result.diet)
        print("Ingredients:", checklist_result.ingredients)
        print("Confirmed allergens:", checklist_result.allergens)
        print("Possible allergens:", checklist_result.possible_allergens)
        print("Preparation time:", checklist_result.prep_time_minutes)
        print("Difficulty:", checklist_result.difficulty)
        print("Selected allergen:", checklist_result.main_allergen)

Response time: 2.67 seconds
Finish reason: stop
Input tokens: 2192
Output tokens: 275

Response passed schema validation.
Diet: vegan
Ingredients: ['cashews', 'tomatoes', 'water', 'vegetable oil', 'cumin', 'salt']
Confirmed allergens: ['nuts']
Possible allergens: []
Preparation time: 25
Difficulty: easy
Selected allergen: nuts


In [15]:
if checklist_result is not None:
    print("--- Proposed substitution ---")

    if checklist_result.substitution is not None:
        print(checklist_result.substitution.model_dump_json(indent=2))
    else:
        print("No substitution proposed.")

    print("\n--- Notes ---")
    for note in checklist_result.notes:
        print("-", note)
else:
    print("No validated result. Review the previous cell's output.")

--- Proposed substitution ---
{
  "replacements": [
    {
      "ingredient": "cashews",
      "replacement": "sunflower seeds",
      "adjustment": "Use 40 g sunflower seeds in place of the 40 g cashews; soak for about 20 minutes as written, then blend with the tomatoes and water as before."
    }
  ],
  "explanation": "The confirmed allergen is nuts, and the selected source is cashews. Replacing the cashews removes the only confirmed nut ingredient while keeping the recipe vegetarian and vegan. Sunflower seeds are used as a practical substitution for texture; this introduces seeds (not one of the listed top-level allergens) and may change flavor and thickness slightly. The remaining original ingredients are tomatoes, water, vegetable oil, cumin, and salt.",
  "remaining_allergens": [],
  "introduced_allergens": []
}

--- Notes ---
- Preparation time and difficulty are explicitly stated in the recipe.
- The target allergen 'nuts' is confirmed present via cashews.
- No possible allerge

### Initial Comparison: One Development Recipe

All three prompting techniques produced schema-valid responses.

- Zero-shot misclassified the diet and incorrectly reported sesame
  as an introduced allergen.
- Few-shot corrected those errors but gave an unclear quantity
  instruction: "same weight by volume".
- Checklist prompting corrected those errors and provided explicit
  quantity and preparation instructions.

Decision: checklist is the strongest response on this example.
Do not select an overall winner until all development recipes
have been evaluated.

These exploratory responses are retained as observations.

## 13. Select a Small Development Evaluation Set

### WHAT
Select six development recipes covering different assessment cases.

### WHY
The brief does not specify a dataset size. A small, varied set keeps
the assessment manageable while demonstrating the required workflow.

### HOW
Select recipes by fixed IDs before running the comparison.
Use exactly the same six recipes for all three prompting techniques.

Keep few-shot examples separate and leave test.json unopened.

### EXPECTED RESULT
Six recipes, giving 18 recipe-analysis requests across three techniques.

### DECISION
Use this fixed subset for the comparison. Report the small sample
size as a limitation and do not claim broad reliability.

In [16]:
selected_ids = [
    "DEVELOPMENT_01",  # Cashew gravy: vegan classification and nuts.
    "DEVELOPMENT_02",  # Egg fried rice: workable egg replacement.
    "DEVELOPMENT_09",  # Mashed potato: two sources of milk.
    "DEVELOPMENT_10",  # Cookies: nuts removed, wheat remains.
    "DEVELOPMENT_11",  # Aubergine dip: missing information, no allergen.
    "DEVELOPMENT_12",  # Risotto: uncertain stock ingredients.
]

# Look up records by ID without modifying the original dataset.
recipes_by_id = {
    recipe["recipe_id"]: recipe
    for recipe in development_recipes
}

evaluation_recipes = [
    recipes_by_id[recipe_id]
    for recipe_id in selected_ids
]

print("Recipes selected:", len(evaluation_recipes))
print("Planned comparison requests:", len(evaluation_recipes) * 3)

for recipe in evaluation_recipes:
    print(recipe["recipe_id"], "-", recipe["reference"]["title"])

Recipes selected: 6
Planned comparison requests: 18
DEVELOPMENT_01 - Cashew tomato gravy
DEVELOPMENT_02 - Vegetable egg fried rice
DEVELOPMENT_09 - Butter cream mashed potato
DEVELOPMENT_10 - Almond wheat cookies
DEVELOPMENT_11 - Roasted aubergine dip
DEVELOPMENT_12 - Vegetarian stock risotto


## 14. Create a Reusable Experiment Function

### WHAT
Create one function to call Azure, validate the response and record
the outcome.

### WHY
All three prompting techniques should use the same execution and
validation process. This also avoids repeating code in the notebook.

### HOW
Store the function in experiments.py.

For each request, record:
- Original response and validated output.
- Validation or API failure status.
- Response time and token usage.
- Finish reason.

Keep failed attempts in our results. Do not automatically repair
responses or retry requests.

### EXPECTED RESULT
A reusable function ready for the 18-request comparison.
Defining or importing the function does not call Azure.

### DECISION
Confirm the import works before running the comparison.

In [17]:
from experiments import run_experiment

print("Experiment function imported successfully.")
print("No Azure requests made in this step.")

Experiment function imported successfully.
No Azure requests made in this step.


## 15. Compare Three Prompting Techniques on Six Recipes

### WHAT
Run zero-shot, few-shot and checklist prompting on the same six recipes.

### WHY
A shared evaluation set gives us a broader comparison than one example.

### HOW
Use the same Azure deployment, output schema and experiment function.
Save the exact requests and each result locally.

Keep all attempts, including failures. Do not automatically retry them.
Rotate technique order across recipes to reduce a fixed ordering effect.

These are 18 new requests. Earlier single-recipe runs remain exploratory.

### EXPECTED RESULT
18 saved attempt records, with validation status, responses, timing
and token usage.

### DECISION
Review completion and validation status first, then score correctness.
The test set remains unused.

In [18]:
import hashlib
import json
from pathlib import Path

import pandas as pd

# Build every request before starting any API calls.
techniques = ["zero_shot", "few_shot", "checklist"]
planned_requests = []

for index, recipe in enumerate(evaluation_recipes):
    recipe_input = recipe["input"]

    messages_by_technique = {
        "zero_shot": prompts.build_zero_shot_messages(**recipe_input),
        "few_shot": prompts.build_few_shot_messages(
            **recipe_input,
            examples=few_shot_examples,
        ),
        "checklist": prompts.build_checklist_messages(**recipe_input),
    }

    # Rotate the order: each technique gets a turn at running first.
    offset = index % len(techniques)
    technique_order = techniques[offset:] + techniques[:offset]

    for technique in technique_order:
        planned_requests.append({
            "recipe_id": recipe["recipe_id"],
            "technique": technique,
            "messages": messages_by_technique[technique],
        })

# Identify this run by its deployment, schema, requests and reference data.
# Changed inputs create a separate results folder.
run_definition = {
    "deployment": deployment,
    "schema": RecipeAnalysis.model_json_schema(),
    "requests": planned_requests,
    "evaluation_records": evaluation_recipes,
}

definition_text = json.dumps(
    run_definition,
    sort_keys=True,
    ensure_ascii=False,
    indent=2,
)

run_id = hashlib.sha256(definition_text.encode("utf-8")).hexdigest()[:12]

results_dir = Path("outputs") / f"development_comparison_{run_id}"
results_dir.mkdir(parents=True, exist_ok=True)

(results_dir / "run_definition.json").write_text(
    definition_text,
    encoding="utf-8",
)

results_path = results_dir / "results.jsonl"

# Reload saved attempts if this exact comparison has already started.
comparison_results = []

if results_path.exists():
    with results_path.open("r", encoding="utf-8") as file:
        comparison_results = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

completed = {
    (result["recipe_id"], result["technique"])
    for result in comparison_results
}

print("Results folder:", results_dir)
print("Previously saved attempts:", len(completed))

for request in planned_requests:
    key = (request["recipe_id"], request["technique"])

    if key in completed:
        continue

    result = run_experiment(
        client=client,
        deployment=deployment,
        messages=request["messages"],
    )

    result["recipe_id"] = request["recipe_id"]
    result["technique"] = request["technique"]

    # Save immediately, before proceeding to the next request.
    with results_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

    comparison_results.append(result)
    completed.add(key)

    print(
        f"{len(completed)}/18 | "
        f"{result['recipe_id']} | "
        f"{result['technique']} | "
        f"{result['status']}"
    )

# This table checks completion and structure, not answer accuracy.
comparison_overview = pd.DataFrame(comparison_results)

display(
    comparison_overview[
        [
            "recipe_id",
            "technique",
            "status",
            "response_seconds",
            "input_tokens",
            "output_tokens",
        ]
    ]
)

Results folder: outputs\development_comparison_a7418792c152
Previously saved attempts: 0
1/18 | DEVELOPMENT_01 | zero_shot | valid
2/18 | DEVELOPMENT_01 | few_shot | valid
3/18 | DEVELOPMENT_01 | checklist | valid
4/18 | DEVELOPMENT_02 | few_shot | valid
5/18 | DEVELOPMENT_02 | checklist | valid
6/18 | DEVELOPMENT_02 | zero_shot | valid
7/18 | DEVELOPMENT_09 | checklist | valid
8/18 | DEVELOPMENT_09 | zero_shot | valid
9/18 | DEVELOPMENT_09 | few_shot | valid
10/18 | DEVELOPMENT_10 | zero_shot | valid
11/18 | DEVELOPMENT_10 | few_shot | valid
12/18 | DEVELOPMENT_10 | checklist | valid
13/18 | DEVELOPMENT_11 | few_shot | valid
14/18 | DEVELOPMENT_11 | checklist | valid
15/18 | DEVELOPMENT_11 | zero_shot | valid
16/18 | DEVELOPMENT_12 | checklist | valid
17/18 | DEVELOPMENT_12 | zero_shot | validation_error
18/18 | DEVELOPMENT_12 | few_shot | valid


,recipe_id,technique,status,response_seconds,input_tokens,output_tokens
0,DEVELOPMENT_01,zero_shot,valid,2.922,2002,248
1,DEVELOPMENT_01,few_shot,valid,1.991,2580,195
2,DEVELOPMENT_01,checklist,valid,2.708,2192,280
3,DEVELOPMENT_02,few_shot,valid,2.498,2574,234
4,DEVELOPMENT_02,checklist,valid,1.917,2186,242
5,DEVELOPMENT_02,zero_shot,valid,2.230,1996,282
6,DEVELOPMENT_09,checklist,valid,2.411,2179,285
7,DEVELOPMENT_09,zero_shot,valid,2.452,1989,261
8,DEVELOPMENT_09,few_shot,valid,2.085,2567,217
9,DEVELOPMENT_10,zero_shot,valid,2.414,1996,271


## 16. Inspect the Validation Failure

### WHAT
Inspect the saved response that failed Pydantic validation.

### WHY
Identify which field or value broke the schema before deciding
whether any change is needed.

### HOW
Display the recorded validation error and original response.
Do not call Azure again, repair the answer or remove the failure.

### EXPECTED RESULT
An explanation of the schema violation based on the actual error.

### DECISION
Document the failure and retain it in the comparison results.

In [19]:
failed_results = [
    result
    for result in comparison_results
    if result["status"] != "valid"
]

for result in failed_results:
    print("Recipe:", result["recipe_id"])
    print("Technique:", result["technique"])
    print("Status:", result["status"])

    print("\n--- Recorded error ---")
    print(result["error"])

    print("\n--- Original response ---")
    print(result["raw_response"])

Recipe: DEVELOPMENT_12
Technique: zero_shot
Status: validation_error

--- Recorded error ---
1 validation error for RecipeAnalysis
notes
  Field required [type=missing, input_value={'title': 'Vegetarian sto...roduced_allergens': []}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

--- Original response ---
{"title":"Vegetarian stock risotto","ingredients":["arborio rice","vegetarian stock","onion","olive oil"],"allergens":["cereals_containing_gluten"],"possible_allergens":["celery","milk","soya"],"diet":"vegetarian","prep_time_minutes":10,"difficulty":"medium","main_allergen":"cereals_containing_gluten","substitution":{"replacements":[{"ingredient":"arborio rice","replacement":"gluten-free risotto rice or certified gluten-free arborio-style rice","adjustment":"Use the same quantity, 100 g, and cook as directed."},{"ingredient":"vegetarian stock","replacement":"gluten-free vegetarian stock","adjustment":"Replace the full 500 ml with an equ

## 17. Evaluate Extraction Correctness

### WHAT
Compare confirmed allergens, diet, preparation time and difficulty
against the reference labels.

### WHY
Schema-valid responses can still contain incorrect information.

### HOW
Use exact comparisons for these standardised fields.
Allergen list order does not matter.

Keep all six attempts per technique in the denominator.
A response that fails validation receives no successful field scores
because it cannot be accepted by our application.

Null matches null when information is genuinely unstated.

### EXPECTED RESULT
A per-recipe score table and correct-answer counts out of six
for each prompting technique.

### DECISION
Use these results alongside substitution judging and resource usage.
These scores do not yet evaluate ingredients, possible-allergen
reasoning or substitution quality.

In [20]:
references_by_id = {
    recipe["recipe_id"]: recipe["reference"]
    for recipe in evaluation_recipes
}

score_rows = []

for result in comparison_results:
    reference = references_by_id[result["recipe_id"]]
    valid = result["status"] == "valid"
    prediction = result["validated_output"] if valid else {}

    score_rows.append({
        "recipe_id": result["recipe_id"],
        "technique": result["technique"],
        "schema_valid": valid,

        # Sets ignore list order.
        "allergens_correct": valid and (
            set(prediction["allergens"]) == set(reference["allergens"])
        ),
        "diet_correct": valid and (
            prediction["diet"] == reference["diet"]
        ),
        "prep_time_correct": valid and (
            prediction["prep_time_minutes"] == reference["prep_time_minutes"]
        ),
        "difficulty_correct": valid and (
            prediction["difficulty"] == reference["difficulty"]
        ),
    })

extraction_scores = pd.DataFrame(score_rows)

print("--- Per-recipe results: True means correct ---")
display(extraction_scores)

score_columns = [
    "schema_valid",
    "allergens_correct",
    "diet_correct",
    "prep_time_correct",
    "difficulty_correct",
]

summary_counts = (
    extraction_scores
    .groupby("technique")[score_columns]
    .sum()
    .astype(int)
)

attempt_counts = extraction_scores.groupby("technique").size()
assert attempt_counts.eq(6).all(), "Expected six attempts per technique."

print("--- Successful results out of 6 per technique ---")
display(summary_counts)

# Save the scores beside the original experiment results.
extraction_scores.to_json(
    results_dir / "extraction_scores.json",
    orient="records",
    indent=2,
)

--- Per-recipe results: True means correct ---


,recipe_id,technique,schema_valid,allergens_correct,diet_correct,prep_time_correct,difficulty_correct
0,DEVELOPMENT_01,zero_shot,True,True,True,True,True
1,DEVELOPMENT_01,few_shot,True,True,True,True,True
2,DEVELOPMENT_01,checklist,True,True,True,True,True
3,DEVELOPMENT_02,few_shot,True,True,True,True,True
4,DEVELOPMENT_02,checklist,True,True,True,True,True
5,DEVELOPMENT_02,zero_shot,True,True,True,True,True
6,DEVELOPMENT_09,checklist,True,True,True,True,True
7,DEVELOPMENT_09,zero_shot,True,True,True,True,True
8,DEVELOPMENT_09,few_shot,True,True,True,True,True
9,DEVELOPMENT_10,zero_shot,True,True,False,True,True


--- Successful results out of 6 per technique ---


,schema_valid,allergens_correct,diet_correct,prep_time_correct,difficulty_correct
technique,,,,,
checklist,6,6,6,6,6
few_shot,6,6,6,6,6
zero_shot,5,5,4,5,5


In [21]:
print("Successful results out of 6 per technique:")
print(summary_counts.to_string())

Successful results out of 6 per technique:
           schema_valid  allergens_correct  diet_correct  prep_time_correct  difficulty_correct
technique                                                                                      
checklist             6                  6             6                  6                   6
few_shot              6                  6             6                  6                   6
zero_shot             5                  5             4                  5                   5


## 18. Define the LLM Judge

### WHAT
Use a separate LLM call to review the proposed substitution.

### WHY
A correctly extracted recipe can still have an unsuitable substitution.
Exact text matching is inappropriate because several replacements
may be valid.

### HOW
Give the judge the original recipe, requested allergen and candidate
answer. Do not provide the technique name or reference substitution.

Use a fixed rubric:
- Was proposing or withholding a substitution appropriate?
- Does the plan remove every source of the selected allergen?
- Does the dish remain workable with the stated adjustments?
- Does the adaptation remain vegetarian?
- Are remaining and introduced allergens reported correctly?

Use yes, no or uncertain. Use not_applicable for substitution-specific
checks when no substitution is proposed.

### EXPECTED RESULT
A structured assessment with a short supporting explanation.

### DECISION
Combine judge assessments with extraction scores and manually review
examples. A judge approval does not certify allergy safety.

In [22]:
import schemas

# Refresh edited modules. Existing experiment results remain unchanged.
importlib.reload(schemas)
importlib.reload(prompts)

from schemas import JudgeAssessment

print("Judge schema and prompt loaded.")
print("No Azure calls made.")

Judge schema and prompt loaded.
No Azure calls made.


## 19. Trial the LLM Judge

### WHAT
Ask the judge to review one saved checklist response.

### WHY
Check that the judge request and output validation work before
reviewing the other results.

### HOW
Use the original mashed-potato recipe and its saved candidate answer.
Do not send the technique name or reference answer.

Save the judge's original response and assessment without changing
the candidate answer. This makes one additional Azure request.

### EXPECTED RESULT
A structured review with an explanation addressing both butter
and cream, substitution practicality and allergen reporting.

### DECISION
Inspect whether the judge's verdict agrees with its explanation.
A schema-valid judge response can still be factually wrong.

In [23]:
# Select a saved response; do not regenerate the recipe analysis.
judge_candidate = next(
    result
    for result in comparison_results
    if result["recipe_id"] == "DEVELOPMENT_09"
    and result["technique"] == "checklist"
)

assert judge_candidate["status"] == "valid"

original_input = recipes_by_id["DEVELOPMENT_09"]["input"]

judge_messages = prompts.build_judge_messages(
    recipe_input=original_input,
    candidate=judge_candidate["validated_output"],
)

# Display the proposed changes so we can assess the judge's reasoning.
print("--- Candidate substitution ---")
print(json.dumps(
    judge_candidate["validated_output"]["substitution"],
    indent=2,
))

start_time = perf_counter()

judge_response = client.with_options(
    max_retries=0,
).chat.completions.create(
    model=deployment,
    messages=judge_messages,
)

judge_seconds = perf_counter() - start_time
judge_raw = judge_response.choices[0].message.content
judge_assessment = None

# Save the request and original response before validating.
judge_trial = {
    "recipe_id": judge_candidate["recipe_id"],
    "technique": judge_candidate["technique"],
    "deployment": deployment,
    "messages": judge_messages,
    "raw_response": judge_raw,
    "response_seconds": round(judge_seconds, 3),
    "finish_reason": judge_response.choices[0].finish_reason,
    "usage": (
        judge_response.usage.model_dump()
        if judge_response.usage is not None
        else None
    ),
}

judge_trial_path = results_dir / "judge_trial.json"

# Use a new filename if a previous trial exists.
if judge_trial_path.exists():
    from uuid import uuid4
    judge_trial_path = results_dir / f"judge_trial_{uuid4().hex[:8]}.json"

judge_trial_path.write_text(
    json.dumps(judge_trial, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

if (
    judge_response.choices[0].finish_reason != "stop"
    or not judge_raw
    or getattr(judge_response.choices[0].message, "refusal", None)
):
    print("\nNo complete judge answer. Inspect the saved response.")
else:
    try:
        judge_assessment = JudgeAssessment.model_validate_json(judge_raw)
    except ValidationError as error:
        print("\nJudge response failed validation:")
        print(error)
        print("\nOriginal response:", judge_raw)
    else:
        print("\nJudge response passed schema validation.")
        print(judge_assessment.model_dump_json(indent=2))

print(f"\nJudge response time: {judge_seconds:.2f} seconds")
print("Saved:", judge_trial_path)

--- Candidate substitution ---
{
  "replacements": [
    {
      "ingredient": "butter",
      "replacement": "plant-based margarine",
      "adjustment": "Use 15 g plant-based margarine in place of 15 g butter."
    },
    {
      "ingredient": "dairy cream",
      "replacement": "unsweetened oat or soy cooking cream",
      "adjustment": "Use 30 ml dairy-free cooking cream in place of 30 ml dairy cream; mash in as written."
    }
  ],
  "explanation": "The original recipe contains confirmed milk allergens from both butter and dairy cream. Replace both milk-containing ingredients to remove the selected allergen. Potatoes, water, and salt remain unchanged. Note that oat cream would introduce cereals_containing_gluten only if the product contains gluten; soy cream would introduce soya. Check product labels before use.",
  "remaining_allergens": [],
  "introduced_allergens": [
    "cereals_containing_gluten",
    "soya"
  ]
}

Judge response passed schema validation.
{
  "decision_approp

In [24]:
if judge_assessment is not None:
    for field, value in judge_assessment.model_dump().items():
        print(f"{field}: {value}")
else:
    print("No validated judge assessment is available.")

decision_appropriate: yes
allergen_removed: yes
dish_workable: yes
vegetarian_preserved: yes
allergen_reporting_correct: no
explanation: A substitution was appropriate because the original recipe clearly contains milk in both butter and dairy cream. The plan replaces both sources with plant-based margarine and dairy-free cooking cream, so the selected allergen is addressed. The dish remains workable and vegetarian. However, the allergen reporting is inconsistent: it lists introduced allergens as "cereals_containing_gluten" and "soya" based on optional product choices, but those are not confirmed ingredients and cross-contact/unspecified composition should not be treated as definite allergens.


## 20. Evaluate Saved Answers with the LLM Judge

### WHAT
Judge each schema-valid answer using the same fixed rubric.

### WHY
Extraction scores do not measure whether substitutions are workable,
complete or consistently reported.

### HOW
Reuse saved recipe analyses without regenerating them.
Hide technique names and reference answers from judge requests.
Reuse the completed judge trial and save each additional assessment.

Keep failed analyses and failed judge requests visible.
Not-applicable verdicts are not substitution approvals.

### EXPECTED RESULT
Judge assessments for the 17 accepted analyses, with any judge
failures retained.

### DECISION
Review the assessments alongside extraction scores before selecting
the preferred prompt. The judge is supporting evidence, not ground truth.

In [25]:
import experiments

importlib.reload(experiments)

# Build judge requests for schema-valid analyses only.
judge_plan = []

for candidate in comparison_results:
    if candidate["status"] != "valid":
        continue

    messages = prompts.build_judge_messages(
        recipe_input=recipes_by_id[candidate["recipe_id"]]["input"],
        candidate=candidate["validated_output"],
    )

    judge_plan.append({
        "recipe_id": candidate["recipe_id"],
        "technique": candidate["technique"],
        "messages": messages,
    })

# Separate results if the judge instructions, candidates or schema change.
judge_definition = {
    "deployment": deployment,
    "schema": JudgeAssessment.model_json_schema(),
    "requests": judge_plan,
}

judge_definition_text = json.dumps(
    judge_definition,
    sort_keys=True,
    ensure_ascii=False,
    indent=2,
)

judge_run_id = hashlib.sha256(
    judge_definition_text.encode("utf-8")
).hexdigest()[:12]

judge_results_path = results_dir / f"judge_results_{judge_run_id}.jsonl"

(results_dir / f"judge_definition_{judge_run_id}.json").write_text(
    judge_definition_text,
    encoding="utf-8",
)

judge_results = []

if judge_results_path.exists():
    with judge_results_path.open("r", encoding="utf-8") as file:
        judge_results = [json.loads(line) for line in file if line.strip()]

completed_judgements = {
    (result["recipe_id"], result["technique"])
    for result in judge_results
}

for request in judge_plan:
    key = (request["recipe_id"], request["technique"])

    if key in completed_judgements:
        continue

    # Reuse the successful trial only if its exact request matches.
    reuse_trial = (
        judge_assessment is not None
        and judge_trial["deployment"] == deployment
        and judge_trial["messages"] == request["messages"]
        and judge_trial["finish_reason"] == "stop"
    )

    if reuse_trial:
        result = {
            "status": "valid",
            "validated_output": judge_assessment.model_dump(),
            "raw_response": judge_trial["raw_response"],
            "response_seconds": judge_trial["response_seconds"],
            "source": "reused_trial",
        }
    else:
        result = experiments.run_experiment(
            client=client,
            deployment=deployment,
            messages=request["messages"],
            output_model=JudgeAssessment,
        )
        result["source"] = "new_request"

    result["recipe_id"] = request["recipe_id"]
    result["technique"] = request["technique"]

    with judge_results_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

    judge_results.append(result)
    completed_judgements.add(key)

    print(
        f"{len(completed_judgements)}/{len(judge_plan)} | "
        f"{result['recipe_id']} | {result['technique']} | "
        f"{result['status']} | {result['source']}"
    )

print("\nJudge results saved:", judge_results_path)

1/17 | DEVELOPMENT_01 | zero_shot | valid | new_request
2/17 | DEVELOPMENT_01 | few_shot | valid | new_request
3/17 | DEVELOPMENT_01 | checklist | valid | new_request
4/17 | DEVELOPMENT_02 | few_shot | valid | new_request
5/17 | DEVELOPMENT_02 | checklist | valid | new_request
6/17 | DEVELOPMENT_02 | zero_shot | valid | new_request
7/17 | DEVELOPMENT_09 | checklist | valid | reused_trial
8/17 | DEVELOPMENT_09 | zero_shot | valid | new_request
9/17 | DEVELOPMENT_09 | few_shot | valid | new_request
10/17 | DEVELOPMENT_10 | zero_shot | valid | new_request
11/17 | DEVELOPMENT_10 | few_shot | valid | new_request
12/17 | DEVELOPMENT_10 | checklist | valid | new_request
13/17 | DEVELOPMENT_11 | few_shot | valid | new_request
14/17 | DEVELOPMENT_11 | checklist | valid | new_request
15/17 | DEVELOPMENT_11 | zero_shot | valid | new_request
16/17 | DEVELOPMENT_12 | checklist | valid | new_request
17/17 | DEVELOPMENT_12 | few_shot | valid | new_request

Judge results saved: outputs\development_com

## 21. Summarise Judge Assessments

### WHAT
Combine judge verdicts with the status of all 18 original attempts.

### WHY
A valid judge response may reject a substitution. Missing substitutions
must not receive a pass simply because some checks are not applicable.

### HOW
For recipes requiring substitution, require yes for:
- Appropriate decision.
- Allergen removal.
- Dish workability.
- Vegetarian suitability.
- Correct allergen reporting.

For recipes requiring no substitution, require:
- No selected allergen or substitution in the candidate answer.
- Appropriate decision and correct allergen reporting.
- Not-applicable verdicts for substitution-specific checks.

Keep six original attempts per technique in the denominator.
An invalid analysis or failed judge assessment is not a pass.

### EXPECTED RESULT
Per-recipe verdicts and judge-approved counts out of six.

### DECISION
Inspect rejected or uncertain answers before selecting a prompt.
These are LLM-judge approvals, not independently verified correctness.

In [26]:
judgements_by_key = {
    (result["recipe_id"], result["technique"]): result
    for result in judge_results
}

judge_rows = []

for candidate in comparison_results:
    key = (candidate["recipe_id"], candidate["technique"])
    reference = references_by_id[candidate["recipe_id"]]

    judgement = judgements_by_key.get(key)
    judge_valid = judgement is not None and judgement["status"] == "valid"

    verdict = judgement["validated_output"] if judge_valid else {}
    approved = False

    if candidate["status"] == "valid" and judge_valid:
        prediction = candidate["validated_output"]

        common_checks_pass = (
            verdict["decision_appropriate"] == "yes"
            and verdict["allergen_reporting_correct"] == "yes"
        )

        substitution_checks = [
            verdict["allergen_removed"],
            verdict["dish_workable"],
            verdict["vegetarian_preserved"],
        ]

        if reference["substitution_required"]:
            approved = (
                common_checks_pass
                and prediction["main_allergen"] == reference["main_allergen"]
                and prediction["substitution"] is not None
                and all(value == "yes" for value in substitution_checks)
            )
        else:
            approved = (
                common_checks_pass
                and prediction["main_allergen"] is None
                and prediction["substitution"] is None
                and all(
                    value == "not_applicable"
                    for value in substitution_checks
                )
            )

    judge_rows.append({
        "recipe_id": candidate["recipe_id"],
        "technique": candidate["technique"],
        "analysis_status": candidate["status"],
        "decision": verdict.get("decision_appropriate"),
        "removed": verdict.get("allergen_removed"),
        "workable": verdict.get("dish_workable"),
        "vegetarian": verdict.get("vegetarian_preserved"),
        "reporting": verdict.get("allergen_reporting_correct"),
        "judge_approved": approved,
        "explanation": verdict.get("explanation"),
    })

judge_scores = pd.DataFrame(judge_rows)

print("--- Per-recipe verdicts ---")
print(judge_scores.drop(columns="explanation").to_string(index=False))

print("\n--- Judge-approved answers out of 6 ---")
judge_summary = judge_scores.groupby("technique")["judge_approved"].agg(
    approved="sum",
    attempts="count",
)
print(judge_summary.to_string())

judge_scores.to_json(
    results_dir / f"judge_scores_{judge_run_id}.json",
    orient="records",
    indent=2,
)

--- Per-recipe verdicts ---
     recipe_id technique  analysis_status decision        removed       workable     vegetarian reporting  judge_approved
DEVELOPMENT_01 zero_shot            valid      yes            yes            yes            yes uncertain           False
DEVELOPMENT_01  few_shot            valid      yes            yes            yes            yes       yes            True
DEVELOPMENT_01 checklist            valid      yes            yes            yes            yes uncertain           False
DEVELOPMENT_02  few_shot            valid      yes            yes            yes            yes        no           False
DEVELOPMENT_02 checklist            valid      yes            yes            yes            yes       yes            True
DEVELOPMENT_02 zero_shot            valid      yes            yes            yes            yes        no           False
DEVELOPMENT_09 checklist            valid      yes            yes            yes            yes        no           Fa

## 22. Manually Review Judge Objections

### WHAT
Inspect two saved candidate answers and the judge's explanations.

### WHY
An LLM judge can make mistakes or apply the rubric inconsistently.
Its verdicts need human review.

### HOW
Review checklist cashew gravy and few-shot egg fried rice.
Compare the original ingredients, proposed replacements and reported
allergens with the judge's explanation.

### EXPECTED RESULT
Identify whether each objection is supported or remains uncertain.

### DECISION
Document any disagreement without silently changing saved judge scores.
Use this review to qualify the prompt-selection decision.

In [27]:
review_keys = [
    ("DEVELOPMENT_01", "checklist"),
    ("DEVELOPMENT_02", "few_shot"),
]

for recipe_id, technique in review_keys:
    candidate = next(
        result
        for result in comparison_results
        if result["recipe_id"] == recipe_id
        and result["technique"] == technique
    )

    assessment = judgements_by_key[
        (recipe_id, technique)
    ]["validated_output"]

    answer = candidate["validated_output"]

    print(f"\n===== {recipe_id} | {technique} =====")

    print("\nORIGINAL RECIPE:")
    print(recipes_by_id[recipe_id]["input"]["recipe_text"])

    print("\nCANDIDATE ALLERGENS:", answer["allergens"])
    print("POSSIBLE ALLERGENS:", answer["possible_allergens"])

    print("\nSUBSTITUTION:")
    print(json.dumps(answer["substitution"], indent=2))

    print("\nCANDIDATE NOTES:")
    for note in answer["notes"]:
        print("-", note)

    print("\nJUDGE EXPLANATION:")
    print(assessment["explanation"])


===== DEVELOPMENT_01 | checklist =====

ORIGINAL RECIPE:
Cashew tomato gravy
Ingredients: 40 g cashews; 200 g tomatoes; 100 ml water; 1 tsp vegetable oil; 1 tsp cumin; salt.
Preparation time: 25 minutes.
Difficulty: easy.
Method: Soak cashews for 20 minutes during preparation. Blend with tomatoes and water, then simmer with oil, cumin and salt.

CANDIDATE ALLERGENS: ['nuts']
POSSIBLE ALLERGENS: []

SUBSTITUTION:
{
  "replacements": [
    {
      "ingredient": "cashews",
      "replacement": "sunflower seeds",
      "adjustment": "Use 40 g sunflower seeds in place of the 40 g cashews; soak briefly if needed for blending, then blend with the tomatoes and water as directed."
    }
  ],
  "explanation": "The confirmed allergen is nuts, selected from the supplied target allergen. The only clear source is cashews, so replacing them removes the selected allergen from the recipe. The adapted recipe remains vegan. Sunflower seeds are a practical vegetarian replacement, but they introduce a new

In [28]:
print(
    judgements_by_key[
        ("DEVELOPMENT_01", "checklist")
    ]["validated_output"]["explanation"]
)

The recipe’s only explicit nut ingredient is cashews (40 g), so proposing a substitution is appropriate for the requested nuts allergen. Replacing cashews with 40 g sunflower seeds and keeping the same blending/simmering method is culinary plausible. The dish stays vegetarian because the original ingredients are plant-based. However, the reporting is only uncertain: the answer labels sunflower seeds as only a possible new risk, but seeds are not nuts, and the explanation also relies on assumptions about allergen categories and ingredient composition rather than strictly reflecting the recipe.


In [30]:
candidate = next(
    result["validated_output"]
    for result in comparison_results
    if result["recipe_id"] == "DEVELOPMENT_02"
    and result["technique"] == "few_shot"
)

print("Possible allergens:", candidate["possible_allergens"])

for change in candidate["substitution"]["replacements"]:
    print("Replacement:", change["replacement"])
    print("Adjustment:", change["adjustment"])

print("Introduced allergens:", candidate["substitution"]["introduced_allergens"])

Possible allergens: ['cereals_containing_gluten']
Replacement: firm tofu, crumbled
Adjustment: Use about the same volume as the 2 eggs; scramble the tofu in the oil until lightly browned before adding the vegetables and rice.
Introduced allergens: ['soya']


### Manual Review and Prompt Selection

Manual review identified limitations in the LLM judge:

- For checklist cashew gravy, the reporting objection was not convincingly
  supported. Sunflower seeds were not incorrectly classified as nuts,
  and individual seed sensitivity was discussed separately.
- For few-shot egg fried rice, the possible-gluten flag was unsupported
  by the supplied recipe. Reporting soya for the tofu replacement was
  reasonable, despite the judge questioning it.

Original judge scores are preserved. Manual observations are recorded
separately rather than used to overwrite the automated assessments.

Selected approach: structured checklist prompting.

Reasons:
- Schema-valid responses on all six development recipes.
- Correct confirmed allergens, diet, preparation time and difficulty
  on all six recipes.
- Highest recorded judge approval: 4/6, compared with few-shot 3/6
  and zero-shot 2/6.

Limitations:
Six synthetic development recipes and a fallible judge using the same
model deployment provide limited evidence of broader reliability.
Ingredient extraction and possible-allergen reporting were not fully
scored by the extraction metrics.

Next:
Freeze the selected prompt and evaluate it on three held-out recipes.
Do not tune it in response to the final test results.

## 23. Evaluate the Selected Prompt on Held-Out Recipes

### WHAT
Evaluate checklist prompting on three previously unused test recipes.

### WHY
Development results influenced our prompt selection.
Held-out recipes provide a separate, small final check.

### HOW
Fix the test IDs before loading the test data:
- TEST_08: multiple sources of the selected allergen.
- TEST_10: multiple allergens, removing only the selected one.
- TEST_11: no confirmed allergen and missing information.

Save the selected deployment, schema, exact requests and responses.
Send only recipe inputs to Azure, never reference answers.

### EXPECTED RESULT
Three saved attempts, including any failures.
Previously saved attempts are reused if this cell is rerun unchanged.

### DECISION
Report the outcomes honestly. Do not tune the prompt on these results.
Three recipes provide a limited final check, not broad validation.

In [31]:
# Fix the selected IDs before opening the test file.
final_test_ids = ["TEST_08", "TEST_10", "TEST_11"]
selected_technique = "checklist"

with (data_dir / "test.json").open("r", encoding="utf-8") as file:
    test_records = json.load(file)

test_by_id = {
    recipe["recipe_id"]: recipe
    for recipe in test_records
}

final_test_recipes = [
    test_by_id[recipe_id]
    for recipe_id in final_test_ids
]

# Freeze the exact requests before making any calls.
final_requests = [
    {
        "recipe_id": recipe["recipe_id"],
        "messages": prompts.build_checklist_messages(**recipe["input"]),
    }
    for recipe in final_test_recipes
]

final_definition = {
    "technique": selected_technique,
    "deployment": deployment,
    "schema": schemas.RecipeAnalysis.model_json_schema(),
    "requests": final_requests,
    "evaluation_records": final_test_recipes,
}

final_definition_text = json.dumps(
    final_definition,
    sort_keys=True,
    ensure_ascii=False,
    indent=2,
)

final_run_id = hashlib.sha256(
    final_definition_text.encode("utf-8")
).hexdigest()[:12]

final_results_dir = Path("outputs") / f"final_test_{final_run_id}"
final_results_dir.mkdir(parents=True, exist_ok=True)

(final_results_dir / "run_definition.json").write_text(
    final_definition_text,
    encoding="utf-8",
)

final_results_path = final_results_dir / "results.jsonl"

# Resume saved attempts without automatically retrying failures.
final_results = []

if final_results_path.exists():
    with final_results_path.open("r", encoding="utf-8") as file:
        final_results = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

completed_test_ids = {
    result["recipe_id"]
    for result in final_results
}

for request in final_requests:
    if request["recipe_id"] in completed_test_ids:
        continue

    result = experiments.run_experiment(
        client=client,
        deployment=deployment,
        messages=request["messages"],
        output_model=schemas.RecipeAnalysis,
    )

    result["recipe_id"] = request["recipe_id"]
    result["technique"] = selected_technique

    with final_results_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

    final_results.append(result)
    completed_test_ids.add(result["recipe_id"])

    print(
        f"{len(completed_test_ids)}/3 | "
        f"{result['recipe_id']} | {result['status']}"
    )

print("\n--- Final test responses ---")

for result in final_results:
    print(f"\n{result['recipe_id']} | {result['status']}")

    if result["status"] == "valid":
        answer = result["validated_output"]

        print("Diet:", answer["diet"])
        print("Allergens:", answer["allergens"])
        print("Possible allergens:", answer["possible_allergens"])
        print("Prep time:", answer["prep_time_minutes"])
        print("Difficulty:", answer["difficulty"])
        print("Selected allergen:", answer["main_allergen"])
    else:
        print("Error:", result["error"])

print("\nSaved:", final_results_dir)

1/3 | TEST_08 | valid
2/3 | TEST_10 | valid
3/3 | TEST_11 | valid

--- Final test responses ---

TEST_08 | valid
Diet: vegan
Allergens: ['sesame']
Possible allergens: []
Prep time: 10
Difficulty: medium
Selected allergen: sesame

TEST_10 | valid
Diet: vegetarian
Allergens: ['cereals_containing_gluten', 'milk']
Possible allergens: []
Prep time: 10
Difficulty: easy
Selected allergen: cereals_containing_gluten

TEST_11 | valid
Diet: vegan
Allergens: []
Possible allergens: []
Prep time: None
Difficulty: None
Selected allergen: None

Saved: outputs\final_test_6700715e5cce


## 24. Review the Held-Out Test Results

### WHAT
Score the extracted fields and inspect each proposed substitution.

### WHY
Schema validity alone does not establish factual correctness or
substitution quality.

### HOW
Compare the same four extraction fields used during development.
Display substitution details separately to avoid output truncation.

### EXPECTED RESULT
Scores out of three and readable substitution details.

### DECISION
Record the results without changing the selected prompt.
Keep extraction scores separate from substitution assessments.

In [32]:
final_score_rows = []

for result in final_results:
    reference = test_by_id[result["recipe_id"]]["reference"]
    valid = result["status"] == "valid"
    answer = result["validated_output"] if valid else {}

    final_score_rows.append({
        "recipe_id": result["recipe_id"],
        "schema_valid": valid,
        "allergens_correct": valid and (
            set(answer["allergens"]) == set(reference["allergens"])
        ),
        "diet_correct": valid and (
            answer["diet"] == reference["diet"]
        ),
        "prep_time_correct": valid and (
            answer["prep_time_minutes"] == reference["prep_time_minutes"]
        ),
        "difficulty_correct": valid and (
            answer["difficulty"] == reference["difficulty"]
        ),
    })

final_scores = pd.DataFrame(final_score_rows)

print(final_scores.to_string(index=False))

print("\n--- Successful results out of 3 ---")
print(final_scores.drop(columns="recipe_id").sum().to_string())

final_scores.to_json(
    final_results_dir / "extraction_scores.json",
    orient="records",
    indent=2,
)

recipe_id  schema_valid  allergens_correct  diet_correct  prep_time_correct  difficulty_correct
  TEST_08          True               True          True               True                True
  TEST_10          True               True          True               True                True
  TEST_11          True               True          True               True                True

--- Successful results out of 3 ---
schema_valid          3
allergens_correct     3
diet_correct          3
prep_time_correct     3
difficulty_correct    3


In [33]:
for result in final_results:
    print(f"\n===== {result['recipe_id']} =====")

    if result["status"] != "valid":
        print("No validated answer.")
        continue

    answer = result["validated_output"]
    substitution = answer["substitution"]

    if substitution is None:
        print("No substitution proposed.")
    else:
        for change in substitution["replacements"]:
            print(
                f"{change['ingredient']} → {change['replacement']}\n"
                f"Adjustment: {change['adjustment']}"
            )

        print("Remaining allergens:", substitution["remaining_allergens"])
        print("Introduced allergens:", substitution["introduced_allergens"])
        print("Explanation:", substitution["explanation"])

    print("Notes:", answer["notes"])


===== TEST_08 =====
sesame oil → neutral vegetable oil
Adjustment: Use the same amount (1 tbsp) to coat the aubergine before roasting.
sesame seeds → toasted sunflower seeds
Adjustment: Use the same amount (1 tsp) as the topping after roasting.
Remaining allergens: []
Introduced allergens: []
Explanation: The selected allergen sesame appears in two confirmed ingredients, so both must be replaced. Swapping the oil and seeds keeps the recipe vegetarian and avoids sesame, but sunflower seeds introduce nuts only if a product is cross-contaminated; sunflower itself is not one of the listed allergens. The original recipe remains vegan aside from the allergen change.
Notes: ['Target allergen sesame is confirmed present in both sesame oil and sesame seeds.', 'Preparation time and difficulty are explicitly stated in the recipe.', 'No uncertainty about vegetarian suitability; the recipe is vegan.']

===== TEST_10 =====
wheat couscous → gluten-free couscous made from corn or rice
Adjustment: Use

## 25. Serve the Selected Approach Through FastAPI

### WHAT
Create a backend endpoint that accepts recipe text and an optional
target allergen, then returns the analysis and judge assessment.

### WHY
Streamlit needs a reusable backend to perform the LLM calls.
The backend must validate both incoming requests and model responses.

### HOW
Reuse the tested checklist prompt, schemas and experiment function.

Endpoints:
- GET /health: confirms the API process is running.
- POST /analyse: analyses a recipe and separately judges the answer.

An invalid recipe analysis produces an error response.
If judging fails, return the analysis with an explicit unavailable
judge status rather than implying approval.

### EXPECTED RESULT
A locally running API with interactive documentation at /docs.

### DECISION
Test the backend before connecting Streamlit.
A successful recipe request normally makes two Azure calls.

## 27. Create the Streamlit Frontend

### WHAT
Build the recipe input, sample helper and results display.

### WHY
Users need a readable interface rather than raw API responses.

### HOW
Streamlit sends requests to FastAPI. Only FastAPI calls Azure.

Use demo.json for the sample helper.
Make API requests only when Analyse recipe is clicked.
Clear old results when the recipe or target changes.

Display the judge's individual verdicts without presenting them
as an allergy-safety certification.

### EXPECTED RESULT
A working interface connected to the existing backend.

### DECISION
Test sample loading, analysis and result display before Docker packaging.

## 29. Run the Application with Docker Compose

### WHAT
Run FastAPI and Streamlit in separate containers.

### WHY
Package the application so it can run without manually starting
two Python processes.

### HOW
Use one shared Dockerfile with a different startup command per service.
Only the backend receives Azure credentials at runtime.

Streamlit connects to http://backend:8000 over the Compose network.
The frontend starts after the backend process health check passes.

### EXPECTED RESULT
Two running services, with the interface available on port 8501.

### DECISION
Verify container health and one recipe analysis before submission.
Container startup does not establish Azure connectivity.

In [34]:
from importlib.metadata import version
from pathlib import Path

runtime_packages = [
    "fastapi",
    "uvicorn",
    "openai",
    "python-dotenv",
    "pydantic",
    "streamlit",
    "httpx",
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in runtime_packages
)

Path("requirements-runtime.txt").write_text(
    requirements + "\n",
    encoding="utf-8",
)

print(requirements)

PackageNotFoundError: No package metadata was found for fastapi